In [21]:
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:


# Deine exakten Pfade von eben
paths = [
    '../participant_package/Wetterdaten Okt 25 - 23 Feb 26/CTH_Temperatur_VC_Halle3 Okt 25 - 23 Feb 26.csv',
    '../participant_package/Wetterdaten Okt 25 - 23 Feb 26/CTH_Temperatur_Zentralgate  Okt 25 - 23 Feb 26.csv',
    '../participant_package/Wetterdaten Okt 25 - 23 Feb 26/CTH_Wind_VC_Halle3  Okt 25 - 23 Feb 26.csv',
    '../participant_package/Wetterdaten Okt 25 - 23 Feb 26/CTH_Wind_Zentralgate  Okt 25 - 23 Feb 26.csv',
    '../participant_package/Wetterdaten Okt 25 - 23 Feb 26/CTH_Windrichtung_VC_Halle3  Okt 25 - 23 Feb 26.csv',
    '../participant_package/Wetterdaten Okt 25 - 23 Feb 26/CTH_Windrichtung_Zentralgate  Okt 25 - 23 Feb 26.csv'
]

# Wir laden sie direkt in einzelne Variablen
try:
    df_temp_halle3    = pd.read_csv(paths[0], sep=';', decimal=',')
    df_temp_gate      = pd.read_csv(paths[1], sep=';', decimal=',')
    df_wind_halle3    = pd.read_csv(paths[2], sep=';', decimal=',')
    df_wind_gate      = pd.read_csv(paths[3], sep=';', decimal=',')
    df_winddir_halle3 = pd.read_csv(paths[4], sep=';', decimal=',')
    df_winddir_gate   = pd.read_csv(paths[5], sep=';', decimal=',')
    
    print("✅ Endlich! Alle 6 Dateien erfolgreich geladen.")
except Exception as e:
    print(f"❌ Da ist immer noch ein Wurm drin: {e}")

# Kurzer Check für dich
df_temp_gate.head(2)

✅ Endlich! Alle 6 Dateien erfolgreich geladen.


,ConfigID,UtcTimestamp,TimeResolution,Quality,Value
0,43805,2025-09-24 10:00:50.943,40.0,192,13.2
1,43805,2025-10-10 09:59:44.277,40.0,192,14.7


In [23]:
# Eine kleine Liste mit Namen und DataFrames für die Schleife
dfs_to_show = [
    ("Temperatur Halle 3", df_temp_halle3),
    ("Temperatur Zentralgate", df_temp_gate),
    ("Windgeschwindigkeit Halle 3", df_wind_halle3),
    ("Windgeschwindigkeit Zentralgate", df_wind_gate),
    ("Windrichtung Halle 3", df_winddir_halle3),
    ("Windrichtung Zentralgate", df_winddir_gate)
]

for title, df in dfs_to_show:
    print(f"--- {title} ---")
    # Zeigt die ersten 3 Zeilen und die Datentypen an
    df.head(5)
    print(f"Shapes: {df.shape} | Spalten: {df.columns.tolist()}\n")

--- Temperatur Halle 3 ---
Shapes: (168479, 5) | Spalten: ['ConfigID', 'UtcTimestamp', 'TimeResolution', 'Quality', 'Value']

--- Temperatur Zentralgate ---
Shapes: (67022, 5) | Spalten: ['ConfigID', 'UtcTimestamp', 'TimeResolution', 'Quality', 'Value']

--- Windgeschwindigkeit Halle 3 ---
Shapes: (3451733, 5) | Spalten: ['ConfigID', 'UtcTimestamp', 'TimeResolution', 'Quality', 'Value']

--- Windgeschwindigkeit Zentralgate ---
Shapes: (3405597, 5) | Spalten: ['ConfigID', 'UtcTimestamp', 'TimeResolution', 'Quality', 'Value']

--- Windrichtung Halle 3 ---
Shapes: (3231544, 5) | Spalten: ['ConfigID', 'UtcTimestamp', 'TimeResolution', 'Quality', 'Value']

--- Windrichtung Zentralgate ---
Shapes: (3189530, 5) | Spalten: ['ConfigID', 'UtcTimestamp', 'TimeResolution', 'Quality', 'Value']



In [24]:
import pandas as pd

# Dictionary für die Übersicht
weather_dfs = {
    "Temperatur Halle 3": df_temp_halle3,
    "Temperatur Zentralgate": df_temp_gate,
    "Wind Halle 3": df_wind_halle3,
    "Wind Zentralgate": df_wind_gate,
    "Windrichtung Halle 3": df_winddir_halle3,
    "Windrichtung Zentralgate": df_winddir_gate
}

exploration_results = []

for name, df in weather_dfs.items():
    # 1. Sicherstellen, dass UtcTimestamp ein Datetime-Objekt ist
    df['UtcTimestamp'] = pd.to_datetime(df['UtcTimestamp'])
    
    # 2. Zeiträume berechnen
    start = df['UtcTimestamp'].min()
    end = df['UtcTimestamp'].max()
    duration = end - start
    count = len(df)
    
    exploration_results.append({
        "Feature": name,
        "Start (UTC)": start,
        "Ende (UTC)": end,
        "Tage": duration.days,
        "Messwerte": count
    })

# Als schöne Tabelle ausgeben
df_summary = pd.DataFrame(exploration_results)
display(df_summary)

# Check auf Lücken (Gaps)
print("\n--- Analyse der zeitlichen Abstände ---")
for name, df in weather_dfs.items():
    # Berechne den Abstand zwischen den Messwerten in Minuten
    time_diffs = df['UtcTimestamp'].sort_values().diff().dt.total_seconds() / 60
    median_diff = time_diffs.median()
    print(f"{name}: Messung alle {median_diff:.0f} Minuten (Median)")

,Feature,Start (UTC),Ende (UTC),Tage,Messwerte
0,Temperatur Halle 3,2025-09-24 10:00:50.943,2026-02-23 13:59:27.817,152,168479
1,Temperatur Zentralgate,2025-09-24 10:00:50.943,2026-02-23 13:57:47.977,152,67022
2,Wind Halle 3,2025-09-24 10:00:50.943,2026-02-23 14:03:31.817,152,3451733
3,Wind Zentralgate,2025-09-24 10:00:50.943,2026-02-23 13:48:00.983,152,3405597
4,Windrichtung Halle 3,2025-11-01 00:00:01.853,2026-02-23 14:06:50.823,114,3231544
5,Windrichtung Zentralgate,2025-11-01 00:00:01.960,2026-02-23 14:01:10.977,114,3189530



--- Analyse der zeitlichen Abstände ---
Temperatur Halle 3: Messung alle 0 Minuten (Median)
Temperatur Zentralgate: Messung alle 1 Minuten (Median)
Wind Halle 3: Messung alle 0 Minuten (Median)
Wind Zentralgate: Messung alle 0 Minuten (Median)
Windrichtung Halle 3: Messung alle 0 Minuten (Median)
Windrichtung Zentralgate: Messung alle 0 Minuten (Median)


In [1]:
import pandas as pd

# Falls du die Datei hochgeladen hast, ersetze den Pfad durch den Dateinamen
file_path = 'open-meteo-53.56N10.00E11m.csv'

# Erster Versuch: Header automatisch finden (meistens beginnt die Tabelle bei 'time')
try:
    # Wir lesen die ersten Zeilen, um zu sehen, wo die Daten anfangen
    with open(file_path, 'r') as f:
        lines = f.readlines(1000) # erste 1000 Zeichen
        skip = 0
        for i, line in enumerate(lines):
            if line.startswith('time'):
                skip = i
                break
    
    df = pd.read_csv(file_path, skiprows=skip)
    print("Daten erfolgreich geladen!")
    print(df.head())
except Exception as e:
    print(f"Fehler beim Laden: {e}")

Fehler beim Laden: [Errno 2] No such file or directory: 'open-meteo-53.56N10.00E11m.csv'
